In [ ]:
# ====================== Kaggle 2xT4 + LlamaFactory + LDOT 全流程训练脚本 ======================
#
# 说明：
# 1. 先在 Kaggle Notebook 设置里开启 Internet
# 2. 训练数据已经在 /kaggle/input/datasets/skye98/ldot-dataset
# 3. 基座模型让 LlamaFactory 自动从 Hugging Face 下载
# 4. 这套脚本默认跑 LDOT 的 Kaggle 双 T4 配置

import os
import json
import torch

# ---------- 路径与环境变量 ----------
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/working/hf_cache/hub"

LLAMAFACTORY_DIR = "/kaggle/working/LlamaFactory"
LDOT_DIR = "/kaggle/working/ldot"
MODEL_PATH = "Qwen/Qwen3.5-0.8B-Base"
DATASET_ROOT = "/kaggle/input/datasets/skye98/ldot-dataset"
OUTPUT_DIR = "/kaggle/working/lf_output/ldot_t4x2"

# 如果你需要 Hugging Face Token，取消下面两行注释
# from huggingface_hub import login
# login(token="你的_HF_TOKEN")

# ---------- 第一步：拉取 LlamaFactory 并安装依赖 ----------
!rm -rf {LLAMAFACTORY_DIR}
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git {LLAMAFACTORY_DIR}
%cd {LLAMAFACTORY_DIR}
!pip uninstall -y torch torchvision torchaudio
!pip install --no-cache-dir torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install -e .
!pip install -r requirements/metrics.txt

# ---------- 第二步：拉取 LDOT 仓库 ----------
%cd /kaggle/working
!rm -rf {LDOT_DIR}
!git clone https://github.com/skye-z/ldot.git {LDOT_DIR}

# ---------- 第三步：检查 GPU ----------
if torch.cuda.is_available():
    print("GPU数量:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i),
            "| 显存:",
            round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),
            "GB",
        )
else:
    print("⚠️ 未检测到 GPU")
print("Torch:", torch.__version__)

# ---------- 第四步：把训练数据和 Kaggle 专用配置导入 LlamaFactory ----------
!python {LDOT_DIR}/scripts/export_kaggle_llamafactory_assets.py \
  --llamafactory-dir {LLAMAFACTORY_DIR} \
  --model-path "{MODEL_PATH}" \
  --dataset-root "{DATASET_ROOT}" \
  --output-dir "{OUTPUT_DIR}"

# ---------- 第五步：检查导出结果 ----------
!sed -n '1,120p' {LLAMAFACTORY_DIR}/setting.kaggle.t4x2.yaml
!grep -n "model_name_or_path" {LLAMAFACTORY_DIR}/setting.kaggle.t4x2.yaml

with open(f"{LLAMAFACTORY_DIR}/data/dataset_info.json", "r", encoding="utf-8") as f:
    dataset_info = json.load(f)
print("dataset_info =", dataset_info)

with open(f"{LLAMAFACTORY_DIR}/data/ldot_train_clean_bilingual_train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
print("train samples =", len(train_data))

# ---------- 第六步：启动双卡训练 ----------
%cd {LLAMAFACTORY_DIR}
!CUDA_VISIBLE_DEVICES=0,1 \
 FORCE_TORCHRUN=1 \
 NPROC_PER_NODE=2 \
 NNODES=1 \
 MASTER_PORT=29501 \
 llamafactory-cli train setting.kaggle.t4x2.yaml

# ---------- 第七步：查看输出 ----------
!find {OUTPUT_DIR} -maxdepth 3 -type f | sort | tail -n 100
